Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn
!pip install pandas
!pip install tqdm
!pip install scikit-image
!pip install scipy
!pip install ace_tools

Calling the Libraries:

In [ ]:
from skimage.feature import local_binary_pattern
from skimage.color import rgb2gray
from skimage import exposure
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import cv2
import glob
import os

Finding Height and Width of an Image:

In [ ]:
import cv2
import os

# Example sample image path from session 1
sample_image_path = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein/vein001_1/01.jpg'

# Load the image in grayscale
img = cv2.imread(sample_image_path, cv2.IMREAD_GRAYSCALE)

# Check if image was loaded successfully
if img is None:
    print("Image could not be loaded. Check the path.")
else:
    # Print its shape
    print("Image shape:", img.shape)

    # Print height and width
    height, width = img.shape
    print("Height:", height)
    print("Width:", width)

Train:

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import normalize


# ==== CONFIG ====
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'


NUM_SUBJECTS = 123
NUM_FINGERS = 4
IMAGE_SIZE = (100, 300)
PATCH_SIZE = 10
stride = 10
LBP_CONFIGS = [(1, 16), (1, 8), (2, 8)]  # (radius, points)


# ----------------------------
# FUNCTION: RIU2 Mapping
# ----------------------------
def get_riu2_mapping(P):
    table = np.zeros(2 ** P, dtype=np.uint8)
    for i in range(2 ** P):
        binary = [(i >> j) & 1 for j in range(P)]
        rotations = [binary[n:] + binary[:n] for n in range(P)]
        min_rotation = min(rotations)
        extended = min_rotation + [min_rotation[0]]
        transitions = sum(extended[j] != extended[j + 1] for j in range(P))
        if transitions <= 2:
            table[i] = sum(min_rotation)
        else:
            table[i] = P + 1
    return table

# ----------------------------
# FUNCTION: Extract LBP histogram
# ----------------------------
def extract_lbp_histogram(block, P, R, riu2_map):
    lbp = local_binary_pattern(block, P, R, method='ror').astype(np.uint16)
    lbp_mapped = riu2_map[lbp]
    hist, _ = np.histogram(lbp_mapped.ravel(), bins=np.arange(0, P + 3), density=True)
    return hist

# ----------------------------
# PRE-COMPUTE MAPPINGS
# ----------------------------
mapping_dict = {P: get_riu2_mapping(P) for _, P in LBP_CONFIGS}

# ----------------------------
# MAIN LOOP (Training: Strategy 1 - Protocol 1)
# ----------------------------
train_lbp_features = []
train_labels = []

print("🚀 Extracting training features (Strategy 1 - Protocol 1)...")

for subject_id in tqdm(range(1, NUM_SUBJECTS + 1)):
    for base_path in [base_path_sess1, base_path_sess2]:
        session = 'session1' if '1st_session' in base_path else 'session2'

        for img_idx in [1, 2, 3, 4, 5]:
            for finger_id in range(1, NUM_FINGERS + 1):
                folder = f"vein{subject_id:03d}_{finger_id}"
                img_path = os.path.join(base_path, folder, f"{img_idx:02d}.jpg")
                print(f"📥 Loading: {img_path}")

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"❌ Missing image: {img_path}")
                    continue

                img = cv2.resize(img, IMAGE_SIZE)
                img = cv2.fastNlMeansDenoising(img, h=10)
                img = cv2.equalizeHist(img).astype(np.float64) / 255.0
                img = (img - np.mean(img)) / (np.std(img) + 1e-8)

                feature_vector = []
                for y in range(0, IMAGE_SIZE[1] - PATCH_SIZE + 1, stride):
                    for x in range(0, IMAGE_SIZE[0] - PATCH_SIZE + 1, stride):
                        block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                        hist = []
                        for R, P in LBP_CONFIGS:
                            mapped = extract_lbp_histogram(block, P, R, mapping_dict[P])
                            hist.extend(mapped)
                        feature_vector.extend(hist)

                if feature_vector:
                    train_lbp_features.append(feature_vector)
                    label = f"{session}_subject{subject_id:03d}_finger{finger_id}_img{img_idx:02d}"
                    train_labels.append(label)

# ----------------------------
# NORMALIZE AND FINALIZE
# ----------------------------
train_lbp_features = np.array(train_lbp_features, dtype=np.float32)
train_lbp_features = normalize(train_lbp_features, norm='l2')
train_labels = np.array(train_labels)

print("\n✅ Training feature extraction complete!")
print("🔢 Feature matrix shape:", train_lbp_features.shape)
print("🟢 Example labels:", train_labels[:5])


Test:

In [ ]:
test_lbp_features = []
test_labels = []

print("🧪 Extracting testing features (Strategy 1 - Protocol 1)...")

for subject_id in tqdm(range(1, NUM_SUBJECTS + 1)):
    for base_path in [base_path_sess1, base_path_sess2]:
        session = 'session1' if '1st_session' in base_path else 'session2'

        for img_idx in [6]:  # Only testing image index
            for finger_id in range(1, NUM_FINGERS + 1):
                folder = f"vein{subject_id:03d}_{finger_id}"
                img_path = os.path.join(base_path, folder, f"{img_idx:02d}.jpg")
                print(f"\n➡️ Subject {subject_id:03d}, Finger {finger_id}, Session: {session}, Image: {img_idx}")
                print(f"  📥 Loading from: {img_path}")

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"  ❌ Missing image: {img_path}")
                    continue

                img = cv2.resize(img, IMAGE_SIZE)
                img = cv2.fastNlMeansDenoising(img, h=10)
                img = cv2.equalizeHist(img).astype(np.float64) / 255.0
                img = (img - np.mean(img)) / (np.std(img) + 1e-8)

                feature_vector = []
                for y in range(0, IMAGE_SIZE[1] - PATCH_SIZE + 1, stride):
                    for x in range(0, IMAGE_SIZE[0] - PATCH_SIZE + 1, stride):
                        block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                        hist = []
                        for R, P in LBP_CONFIGS:
                            mapped = extract_lbp_histogram(block, P, R, mapping_dict[P])
                            hist.extend(mapped)
                        feature_vector.extend(hist)

                if feature_vector:
                    test_lbp_features.append(feature_vector)
                    label = f"{session}_subject{subject_id:03d}_finger{finger_id}_img{img_idx:02d}"
                    test_labels.append(label)

# Finalize
test_lbp_features = np.array(test_lbp_features, dtype=np.float32)
test_lbp_features = normalize(test_lbp_features, norm='l2')
test_labels = np.array(test_labels)

print("✅ Testing feature extraction complete!")
print("🔢 Testing data shape:", test_lbp_features.shape)
print("🟢 Example test labels:", test_labels[:5])


Benchmarking

In [ ]:
# ✅ Helper function to extract subject and finger ID
def extract_subject_and_finger(label):
    parts = label.split('_')
    subject = parts[1]  # e.g., "subject023"
    finger = parts[2]   # e.g., "finger2"
    return subject, finger

correct_matches = 0
total_tests = len(test_lbp_features)

print("\n🔍 Classifying test data using Manhattan distance (Match: Subject ID + Finger)...\n")

for i in range(total_tests):
    test_vec = test_lbp_features[i]
    true_label = test_labels[i]

    # Compute Manhattan distances
    distances = np.sum(np.abs(train_lbp_features - test_vec), axis=1)
    min_index = np.argmin(distances)
    predicted_label = train_labels[min_index]

    # Extract subject and finger
    true_subj, true_finger = extract_subject_and_finger(true_label)
    pred_subj, pred_finger = extract_subject_and_finger(predicted_label)

    # Match if both subject and finger match
    if true_subj == pred_subj and true_finger == pred_finger:
        correct_matches += 1
        match_symbol = "✅"
        result = "CORRECT (person + finger)"
    else:
        match_symbol = "❌"
        result = "WRONG"

    # Logging the result
    print(f"\n🎯 Test sample {i+1}/{total_tests}")
    print(f"    🧾 Predicted: {predicted_label}")
    print(f"    🎯 Actual   : {true_label}")
    print(f"    ➡️  Result   : {match_symbol} {result}")

# 📊 Final Accuracy
accuracy = (correct_matches / total_tests) * 100
print("\n📊 Final Results")
print(f"✅ Correct person+finger matches: {correct_matches} / {total_tests}")
print(f"🎯 Recognition Accuracy (Subject + Finger): {accuracy:.2f}%")


Session Independent R5

In [ ]:
# ✅ Helper function to extract subject and finger ID
def extract_subject_and_finger(label):
    parts = label.split('_')
    subject = parts[1]  # e.g., "subject023"
    finger = parts[2]   # e.g., "finger2"
    return subject, finger

rank1_correct = 0
rank5_correct = 0
total_tests = len(test_lbp_features)

print("\n🔍 Classifying test data using Manhattan distance (Match: Subject ID + Finger)...\n")

for i in range(total_tests):
    test_vec = test_lbp_features[i]
    true_label = test_labels[i]

    # Compute Manhattan distances
    distances = np.sum(np.abs(train_lbp_features - test_vec), axis=1)
    sorted_indices = np.argsort(distances)  # ascending order

    true_subj, true_finger = extract_subject_and_finger(true_label)

    # === Rank-1 Check
    pred_label_r1 = train_labels[sorted_indices[0]]
    pred_subj_r1, pred_finger_r1 = extract_subject_and_finger(pred_label_r1)
    if true_subj == pred_subj_r1 and true_finger == pred_finger_r1:
        rank1_correct += 1
        r1_match = "✅"
    else:
        r1_match = "❌"

    # === Rank-5 Check
    match_found = False
    for k in range(5):  # Top-5
        pred_label = train_labels[sorted_indices[k]]
        pred_subj, pred_finger = extract_subject_and_finger(pred_label)
        if true_subj == pred_subj and true_finger == pred_finger:
            rank5_correct += 1
            match_found = True
            break
    r5_match = "✅" if match_found else "❌"

    # Logging result
    print(f"\n🎯 Test sample {i+1}/{total_tests}")
    print(f"    🎯 Actual         : {true_label}")
    print(f"    🧾 Rank-1 Predict : {pred_label_r1}")
    print(f"    🥇 Rank-1 Match   : {r1_match}")
    print(f"    🖐️  Rank-5 Match   : {r5_match}")

# 📊 Final Accuracy
rank1_accuracy = (rank1_correct / total_tests) * 100
rank5_accuracy = (rank5_correct / total_tests) * 100

print("\n📊 Final Results")
print(f"🥇 Rank-1 Correct Matches: {rank1_correct} / {total_tests}")
print(f"🖐️  Rank-5 Correct Matches: {rank5_correct} / {total_tests}")
print(f"✅ Rank-1 Accuracy: {rank1_accuracy:.2f}%")
print(f"✅ Rank-5 Accuracy: {rank5_accuracy:.2f}%")


Session Independent CMC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Helper function
def extract_subject_and_finger(label):
    parts = label.split('_')
    return parts[1], parts[2]

# === CONFIGURATION ===
max_rank = 100
rank_correct = np.zeros(max_rank)
total_tests = len(test_lbp_features)

print("📊 Calculating Session-Independent CMC Curve (Matching: Subject + Finger)...")

for i in range(total_tests):
    proj_test = test_lbp_features[i]
    true_label = test_labels[i]
    true_subj, true_finger = extract_subject_and_finger(true_label)
    true_id = f"{true_subj}_{true_finger}"

    # Compute Manhattan distances to all training samples
    distances = np.sum(np.abs(train_lbp_features - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    # Find first correct match
    for r in range(max_rank):
        candidate_label = train_labels[sorted_indices[r]]
        cand_subj, cand_finger = extract_subject_and_finger(candidate_label)
        candidate_id = f"{cand_subj}_{cand_finger}"

        if candidate_id == true_id:
            rank_correct[r:] += 1
            break

# Normalize to percentage
cmc_curve = (rank_correct / total_tests) * 100

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, max_rank + 1), cmc_curve, label="Session-Independent CMC", linewidth=2)
plt.xlabel("Rank")
plt.ylabel("Identification Accuracy (%)")
plt.title("Session-Independent CMC Curve — LBP$_{\\mathrm{RIU2}}$((8,1), (16,1), (8,2)) (Strategy 2, Protocol 3)")
plt.grid(True)
plt.xticks(np.arange(0, max_rank + 1, 10))
plt.legend()
plt.tight_layout()
plt.show()

# Print key values
print(f"🎯 Rank-1 Accuracy   : {cmc_curve[0]:.2f}%")
print(f"🎯 Rank-5 Accuracy   : {cmc_curve[4]:.2f}%")
print(f"🎯 Rank-10 Accuracy  : {cmc_curve[9]:.2f}%")
print(f"🎯 Rank-100 Accuracy : {cmc_curve[99]:.2f}%")


Session Independent Precision, Recall, F1, Accuracy

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# ✅ Helper function to extract subject and finger from a label
def extract_subject_and_finger(label):
    parts = label.split('_')
    return parts[1], parts[2]  # subject, finger

# === Initialize scores and labels
all_scores = []
all_labels = []

# === Pairwise comparison (Session-Independent Verification)
for test_idx in range(len(test_lbp_features)):
    test_vec = test_lbp_features[test_idx]
    test_label = test_labels[test_idx]
    test_subj, test_finger = extract_subject_and_finger(test_label)
    test_id = f"{test_subj}_{test_finger}"

    for train_idx in range(len(train_lbp_features)):
        train_vec = train_lbp_features[train_idx]
        train_label = train_labels[train_idx]
        train_subj, train_finger = extract_subject_and_finger(train_label)
        train_id = f"{train_subj}_{train_finger}"

        # ✅ Similarity score: higher = more similar
        score = -np.sum(np.abs(test_vec - train_vec))
        all_scores.append(score)

        # ✅ Label as genuine if subject + finger match
        is_genuine = int(test_id == train_id)
        all_labels.append(is_genuine)

# === Normalize similarity scores to [0, 1]
scores = np.array(all_scores)
labels = np.array(all_labels)
scores = (scores - scores.min()) / (scores.max() - scores.min())

# === Find threshold that maximizes F1 score
best_f1 = best_thresh = best_prec = best_rec = 0

for t in np.linspace(0, 1, 1000):
    preds = (scores >= t).astype(int)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        best_prec = precision
        best_rec = recall

# === Final metrics at best threshold
final_preds = (scores >= best_thresh).astype(int)
accuracy = accuracy_score(labels, final_preds)

# === Output summary
print("🔍 Verification Summary — Matching on Subject + Finger (Session-Independent)")
print(f"📍 Optimal Threshold  : {best_thresh:.3f}")
print(f"✔️ Accuracy           : {accuracy * 100:.2f}%")
print(f"✔️ Precision (PR)     : {best_prec * 100:.2f}%")
print(f"✔️ Recall (RC)        : {best_rec * 100:.2f}%")
print(f"✔️ F1 Score (F1)      : {best_f1 * 100:.2f}%")
